# NBA Scout Data Processing

Notebook này dùng để thử nghiệm data processing trước khi chuyển logic vào codebase local.

Mục tiêu tạo 3 gold datasets:

- `player_role_features.parquet`: role profile và similarity features.
- `performance_training.parquet`: rolling form features và future production targets.
- `salary_training.parquet`: player-season salary analysis/training table.

Notebook cố tình không import code local trong `src/` hoặc `app/` để giữ giai đoạn exploration tách biệt.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# Colab/bootstrap install. nba-api import path is nba_api.
%pip install -q nba-api kagglehub

## 1. Paths and Input Contracts

Notebook ưu tiên đọc data từ Google Drive folder `My Drive/nba-scout-assistant/data`.

Trong Colab, notebook sẽ tự mount Google Drive trước khi resolve path. Sau cell path, output đúng nên là:

```text
Using data folder: /content/drive/MyDrive/nba-scout-assistant/data
```

Nếu chạy local và có Drive sync ở path khác, set env var `NBA_SCOUT_DATA_DIR` trỏ tới folder `data`.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

DRIVE_PROJECT_FOLDER_ID = "1aweR7CoNntwybx3Y9eTAXje2AfV_8K8H"
COLAB_DRIVE_DATA_DIR = Path("/content/drive/MyDrive/nba-scout-assistant/data")
COLAB_LEGACY_DRIVE_DATA_DIR = Path("/content/drive/My Drive/nba-scout-assistant/data")

def running_in_colab() -> bool:
    """Input: none. Output: True when the notebook is running in Google Colab."""
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except ImportError:
        return False

def mount_google_drive_if_needed() -> None:
    """Input: none. Output: mounted /content/drive when running in Colab and Drive data is not visible yet."""
    if not running_in_colab():
        return
    if COLAB_DRIVE_DATA_DIR.exists() or COLAB_LEGACY_DRIVE_DATA_DIR.exists():
        return
    from google.colab import drive  # type: ignore
    print("Mounting Google Drive so the project data folder is visible...")
    drive.mount("/content/drive")

def first_existing_path(paths: list[Path]) -> Path | None:
    """Input: candidate paths. Output: first existing Path or None. Used to locate Drive/local data folder."""
    for path in paths:
        if path.exists():
            return path
    return None

mount_google_drive_if_needed()

env_data_dir = os.getenv("NBA_SCOUT_DATA_DIR")
drive_data_candidates = [
    COLAB_DRIVE_DATA_DIR,
    COLAB_LEGACY_DRIVE_DATA_DIR,
    Path.home() / "Google Drive" / "My Drive" / "nba-scout-assistant" / "data",
    Path.home() / "Library" / "CloudStorage" / "GoogleDrive-MyDrive" / "nba-scout-assistant" / "data",
]

DATA_DIR = Path(env_data_dir).expanduser().resolve() if env_data_dir else first_existing_path(drive_data_candidates)
if DATA_DIR is None and running_in_colab() and env_data_dir is None:
    raise FileNotFoundError(
        "Google Drive is mounted, but the project data folder was not found. "
        f"Expected: {COLAB_DRIVE_DATA_DIR}. "
        f"Drive folder ID checked from connector: {DRIVE_PROJECT_FOLDER_ID}. "
        "Make sure the folder is under My Drive, not only Shared with me, or set NBA_SCOUT_DATA_DIR."
    )
elif DATA_DIR is None:
    DATA_DIR = PROJECT_ROOT / "data"
    print(f"Drive data folder not found. Falling back to local data folder: {DATA_DIR}")
    print("Expected Colab Drive path:", COLAB_DRIVE_DATA_DIR)
else:
    print(f"Using data folder: {DATA_DIR}")

BRONZE_DIR = DATA_DIR / "bronze"
RAW_DIR = DATA_DIR / "raw"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
GOLD_DIR.mkdir(parents=True, exist_ok=True)

PLAYERS_PATH = RAW_DIR / "players.parquet"
PLAYER_BIO_PATH = RAW_DIR / "player_bio" / "eoin_players.csv"
NBA_TRADITIONAL_DIR = RAW_DIR / "nba_traditional"
GAME_LOGS_PATH = SILVER_DIR / "player_game_logs.parquet"
NBA_API_GAME_LOGS_PATH = RAW_DIR / "player_game_logs_nba_api.parquet"
SEASON_STATS_PATH = RAW_DIR / "player_season_stats.parquet"
ADVANCED_STATS_PATH = RAW_DIR / "player_stats_advanced" / "player_stats_advanced_rs.csv"
USAGE_STATS_PATH = RAW_DIR / "player_stats_advanced" / "player_stats_usage_rs.csv"
DEFENSE_STATS_PATH = RAW_DIR / "player_stats_advanced" / "player_stats_defense_rs.csv"
ADVANCED_PATCH_2023_DIR = RAW_DIR / "player_stats_advanced_patch" / "2023_24"
ADVANCED_PATCH_2024_DIR = RAW_DIR / "player_stats_advanced_patch" / "2024_25"
RODNEY_ADVANCED_PATH = ADVANCED_PATCH_2023_DIR / "Advanced.csv"
RATIN_ADVANCED_2024_25_PATH = ADVANCED_PATCH_2024_DIR / "NBA Player Advanced Stats_2024-25.csv"
RATIN_TOTAL_2024_25_PATH = ADVANCED_PATCH_2024_DIR / "NBA Player Stats_2024-25_Total.csv"
SALARY_CAP_PATH = RAW_DIR / "salary_cap" / "salary_cap_by_season.csv"
PLAYER_SEASON_SALARIES_PATH = SILVER_DIR / "player_season_salaries.parquet"

OUTPUT_ROLE_FEATURES = GOLD_DIR / "player_role_features.parquet"
OUTPUT_PERFORMANCE_TRAINING = GOLD_DIR / "performance_training.parquet"
OUTPUT_SALARY_TRAINING = GOLD_DIR / "salary_training.parquet"

critical_input_paths = {
    "players parquet": PLAYERS_PATH,
    "player bio csv": PLAYER_BIO_PATH,
    "season stats parquet": SEASON_STATS_PATH,
    "salary cap csv": SALARY_CAP_PATH,
    "2023-24 advanced patch": RODNEY_ADVANCED_PATH,
    "2024-25 advanced patch": RATIN_ADVANCED_2024_25_PATH,
    "salary parquet": PLAYER_SEASON_SALARIES_PATH,
    "game logs parquet": GAME_LOGS_PATH,
}
for label, path in critical_input_paths.items():
    print(f"{label}: {'FOUND' if path.exists() else 'missing'} - {path}")

RAW_DIR, GOLD_DIR


In [ ]:
RAW_SCHEMAS = {
    "players": [
        "player_id", "player_name", "birth_date", "position", "height", "weight",
    ],
    "player_game_logs": [
        "player_id", "game_date", "game_id", "season", "team_id", "minutes", "points", "assists",
        "rebounds", "offensive_rebounds", "defensive_rebounds", "steals", "blocks", "personal_fouls",
        "turnovers", "true_shooting_pct", "opponent", "home_away", "rest_days",
    ],
    "player_season_stats": [
        "player_id", "season", "team_id", "age", "minutes", "usage_pct", "points_per_100",
        "assists_per_100", "rebounds_per_100", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate",
        "foul_rate", "pace", "possessions", "offensive_rating", "defensive_rating",
    ],
    "salary_cap_by_season": ["season", "salary_cap_usd", "tax_level_usd"],
    "player_season_salaries": [
        "player_name", "team", "season_start_year", "season_end_year", "season_label", "salary_usd",
        "source", "source_file", "collected_at",
    ],
}

def read_table(path: Path) -> pd.DataFrame:
    """Input: CSV/parquet path. Output: DataFrame, or empty DataFrame if file is missing."""
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path.suffix}")

def report_schema(name: str, df: pd.DataFrame, expected_columns: list[str]) -> None:
    """Input: table name, DataFrame, expected columns. Output: printed schema/missing-column report."""
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    missing = sorted(set(expected_columns) - set(df.columns))
    extra = sorted(set(df.columns) - set(expected_columns))
    if missing:
        print("  Missing expected columns:", missing)
    if extra:
        print("  Extra columns:", extra[:25], "..." if len(extra) > 25 else "")


## 2. Load Raw Tables

## 2A. Primary: Fetch Historical Game Logs From KaggleHub

Historical game logs không nên lấy trực tiếp từ `stats.nba.com` qua `nba_api` trong Colab vì dễ timeout/throttle. Nguồn chính cho MVP là Kaggle dataset `szymonjwiak/nba-traditional`, tải một lần bằng KaggleHub rồi chuẩn hóa thành `data/silver/player_game_logs.parquet`.

Notebook không giả định tên file trong dataset. Nó scan các file tabular, tìm candidate có các cột box-score như points/assists/rebounds, rồi chuẩn hóa schema canonical.

In [ ]:
try:
    import kagglehub
    KAGGLEHUB_AVAILABLE = True
except ImportError:
    KAGGLEHUB_AVAILABLE = False
    print("kagglehub is not installed. Run `%pip install kagglehub` in this notebook if you want to fetch Kaggle datasets.")

# nba_api remains available as a fallback for lightweight metadata, not as the primary game-log source.
# Nếu kernel chưa có nba_api, chạy cell install `nba-api` ở đầu notebook rồi restart kernel nếu cần.

try:
    from nba_api.stats.endpoints import leaguedashplayerstats, playergamelogs
    from nba_api.stats.static import players as nba_static_players
    NBA_API_AVAILABLE = True
except ImportError:
    NBA_API_AVAILABLE = False
    print("nba_api is not installed. Run `%pip install nba-api` in this notebook if you want to fetch data.")

In [ ]:
AUTO_FETCH_KAGGLE_TRADITIONAL_IF_MISSING = True
AUTO_FETCH_NBA_API_IF_MISSING = False
OVERWRITE_EXISTING_NBA_API_RAW = False
KAGGLE_TRADITIONAL_DATASET = "szymonjwiak/nba-traditional"
KAGGLE_ADVANCED_2023_DATASET = "rodneycarroll78/nba-stats-1980-2024"
KAGGLE_ADVANCED_2024_DATASET = "ratin21/nba-player-stats-2024-25-per-game"
AUTO_FETCH_ADVANCED_PATCH_IF_MISSING = True

# NBA season format used by stats.nba.com. Keep this aligned with salary data availability.
FETCH_SEASONS = [
    "2016-17", "2017-18", "2018-19", "2019-20", "2020-21",
    "2021-22", "2022-23", "2023-24", "2024-25",
]

# Temporal split for model experiments. Do not random-split these datasets.
TRAIN_END_SEASON = "2021-22"
VALIDATION_SEASONS = ["2022-23"]
TEST_SEASONS = ["2023-24"]
FINAL_HOLDOUT_SEASONS = ["2024-25"]
SEASON_TYPE = "Regular Season"
NBA_API_TIMEOUT_SECONDS = 180
NBA_API_MAX_RETRIES = 3

# NBA Stats can throttle requests. Increase this if calls fail intermittently.
REQUEST_SLEEP_SECONDS = 3.0
NBA_API_CACHE_DIR = RAW_DIR / "nba_api_cache"
NBA_API_CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import re

def to_snake_case(column: str) -> str:
    """Input: raw column name. Output: normalized snake_case column name for schema matching."""
    column = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", str(column))
    column = re.sub(r"[^a-zA-Z0-9]+", "_", column)
    return column.strip("_").lower()

def read_tabular_file(path: Path) -> pd.DataFrame:
    """Input: CSV/parquet file path. Output: DataFrame read with the appropriate pandas reader."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported file type: {path}")

def download_nba_traditional_dataset() -> Path:
    """Input: none. Output: local KaggleHub dataset path for szymonjwiak/nba-traditional."""
    if not KAGGLEHUB_AVAILABLE:
        raise ImportError("Install kagglehub first by running `%pip install kagglehub`, then restart/re-run the notebook.")
    NBA_TRADITIONAL_DIR.mkdir(parents=True, exist_ok=True)
    dataset_path = kagglehub.dataset_download(
        KAGGLE_TRADITIONAL_DATASET,
        output_dir=str(NBA_TRADITIONAL_DIR),
    )
    print("Dataset downloaded to:", dataset_path)
    return Path(dataset_path)

def download_kaggle_dataset(dataset_slug: str, output_dir: Path) -> Path:
    """Input: KaggleHub dataset slug and output dir. Output: downloaded dataset path."""
    if not KAGGLEHUB_AVAILABLE:
        raise ImportError("Install kagglehub first by running `%pip install kagglehub`, then restart/re-run the notebook.")
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset_path = kagglehub.dataset_download(dataset_slug, output_dir=str(output_dir))
    print("Dataset downloaded to:", dataset_path)
    return Path(dataset_path)

def normalize_name_key(value: object) -> str | None:
    """Input: player name. Output: normalized key for cross-source player joins."""
    if pd.isna(value):
        return None
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def normalize_team_abbreviation(value: object) -> str | None:
    """Input: source team abbreviation. Output: abbreviation aligned with game-log team ids."""
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    mapping = {"CHO": "CHA", "BRK": "BKN", "PHO": "PHX", "NOH": "NOP", "NOK": "NOP", "NJN": "BKN"}
    return mapping.get(text, text)

def season_label_from_bref_end_year(value: object) -> str | None:
    """Input: Basketball-Reference season end year. Output: season label such as 2023-24."""
    if pd.isna(value):
        return None
    end_year = int(value)
    return f"{end_year - 1}-{str(end_year)[-2:]}"

def find_tabular_files(data_dir: Path) -> list[Path]:
    """Input: dataset directory. Output: all CSV/parquet files found recursively."""
    return sorted(path for path in data_dir.rglob("*") if path.suffix.lower() in {".csv", ".parquet"})

def find_player_boxscore_file(data_dir: Path) -> Path:
    """Input: raw dataset directory. Output: most likely player-level box-score file path."""
    signal_groups = [
        {"pts", "ast", "reb"},
        {"points", "assists", "rebounds_total"},
        {"points", "assists", "rebounds"},
    ]
    candidates = []
    for path in find_tabular_files(data_dir):
        try:
            sample = read_tabular_file(path)
        except Exception as exc:
            print(f"Could not read {path}: {exc}")
            continue
        normalized_columns = {to_snake_case(column) for column in sample.columns}
        best_match = max((len(group & normalized_columns) for group in signal_groups), default=0)
        player_signals = {"player_id", "playerid", "person_id", "person_name", "player_name", "player"}
        team_only_signals = {"teamid", "team_id"}
        has_player_signal = bool(player_signals & normalized_columns)
        team_only_score = len(team_only_signals & normalized_columns)
        if best_match >= 2 and has_player_signal:
            candidates.append({
                "path": path,
                "shape": sample.shape,
                "columns": sample.columns.tolist(),
                "score": best_match + 2 * int(has_player_signal) - team_only_score,
            })

    for candidate in candidates:
        print("Candidate:", candidate["path"], candidate["shape"])
        print("Columns:", candidate["columns"])
    if not candidates:
        raise FileNotFoundError("No player box-score candidate found in nba-traditional dataset.")
    return sorted(candidates, key=lambda item: (item["score"], item["shape"][0]), reverse=True)[0]["path"]

def season_end_year_from_date(game_date: pd.Timestamp) -> int | None:
    """Input: game date. Output: NBA season end year, e.g. 2016-17 -> 2017."""
    if pd.isna(game_date):
        return None
    return game_date.year + 1 if game_date.month >= 7 else game_date.year

def season_label_from_end_year(end_year: object) -> str | None:
    """Input: season end year. Output: season label such as 2024-25."""
    if pd.isna(end_year):
        return None
    end_year = int(end_year)
    start_year = end_year - 1
    return f"{start_year}-{str(end_year)[-2:]}"

def parse_minutes(value: object) -> float | None:
    """Input: minutes value as number or MM:SS text. Output: minutes as float."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    if ":" in text:
        minutes, seconds = text.split(":", 1)
        return float(minutes) + float(seconds) / 60
    return float(text)

def canonicalize_kaggle_player_game_logs(raw: pd.DataFrame) -> pd.DataFrame:
    """Input: raw Kaggle player box-score DataFrame. Output: canonical silver player_game_logs DataFrame."""
    df = raw.copy()
    df.columns = [to_snake_case(column) for column in df.columns]
    rename_map = {
        "gameid": "game_id",
        "date": "game_date",
        "playerid": "player_id",
        "player": "player_name",
        "person_id": "player_id",
        "person_name": "player_name",
        "game_id": "game_id",
        "game_date": "game_date",
        "team_id": "team_id",
        "team": "team_abbreviation",
        "team_tricode": "team_abbreviation",
        "team_abbreviation": "team_abbreviation",
        "min": "minutes",
        "minutes": "minutes",
        "pts": "points",
        "points": "points",
        "ast": "assists",
        "assists": "assists",
        "reb": "rebounds",
        "rebounds_total": "rebounds",
        "oreb": "offensive_rebounds",
        "dreb": "defensive_rebounds",
        "stl": "steals",
        "blk": "blocks",
        "pf": "personal_fouls",
        "home": "home_team",
        "away": "away_team",
        "tov": "turnovers",
        "turnovers": "turnovers",
        "fgm": "fgm",
        "fga": "fga",
        "3_pm": "fg3m",
        "3_pa": "fg3a",
        "3pm": "fg3m",
        "3pa": "fg3a",
        "fg3m": "fg3m",
        "fg3a": "fg3a",
        "ftm": "ftm",
        "fta": "fta",
    }
    df = df.rename(columns={old: new for old, new in rename_map.items() if old in df.columns})
    if "team_id" not in df.columns and "team_abbreviation" in df.columns:
        df["team_id"] = df["team_abbreviation"]
    required = ["player_id", "game_id", "game_date", "team_id", "minutes", "points", "assists", "rebounds"]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"Missing required player game-log columns: {missing}. Available columns: {df.columns.tolist()}")

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df["minutes"] = df["minutes"].map(parse_minutes)
    numeric_columns = [
        "points", "assists", "rebounds", "offensive_rebounds", "defensive_rebounds",
        "steals", "blocks", "personal_fouls", "turnovers", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta",
    ]
    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    if {"team_abbreviation", "home_team", "away_team"}.issubset(df.columns):
        is_home = df["team_abbreviation"].astype(str).eq(df["home_team"].astype(str))
        df["home_away"] = np.where(is_home, "HOME", "AWAY")
        df["opponent"] = np.where(is_home, df["away_team"], df["home_team"])

    if {"points", "fga", "fta"}.issubset(df.columns):
        denominator = 2 * (df["fga"] + 0.44 * df["fta"]).replace(0, np.nan)
        df["true_shooting_pct"] = df["points"] / denominator

    if "season_end_year" not in df.columns:
        df["season_end_year"] = df["game_date"].map(season_end_year_from_date).astype("Int64")
    df["season_label"] = df["season_end_year"].map(season_label_from_end_year)
    df["season"] = df["season_label"]

    season_type_cols = [column for column in df.columns if "season" in column or "type" in column]
    print("Season/type columns:", season_type_cols)
    for column in season_type_cols:
        if column != "season" and df[column].dtype == "object":
            values = df[column].astype(str).str.lower()
            if values.str.contains("regular").any():
                before = len(df)
                df = df[values.str.contains("regular", na=False)].copy()
                print(f"Filtered regular season using {column}: {before:,} -> {len(df):,}")
                break

    df = df[df["season_end_year"].between(2017, 2025)].copy()
    duplicate_key = ["player_id", "game_id", "team_id"]
    print("Duplicate rows:", df.duplicated(subset=duplicate_key, keep=False).sum())
    display(df[required].isna().mean().sort_values(ascending=False).to_frame("missing_rate"))

    keep_cols = [
        "player_id", "player_name", "game_date", "game_id", "season", "season_end_year", "season_label",
        "team_id", "team_abbreviation", "opponent", "home_away", "minutes", "points", "assists",
        "rebounds", "offensive_rebounds", "defensive_rebounds", "steals", "blocks", "personal_fouls",
        "turnovers", "fgm", "fga", "fg3m", "fg3a", "ftm", "fta", "true_shooting_pct",
    ]
    keep_cols = [column for column in keep_cols if column in df.columns]
    canonical = df[keep_cols].sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    canonical["rest_days"] = canonical.groupby("player_id")["game_date"].diff().dt.days
    return canonical

def build_players_from_game_logs(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical game logs. Output: minimal players table with placeholder bio fields."""
    if game_logs_df.empty or "player_name" not in game_logs_df.columns:
        return pd.DataFrame()
    players_df = game_logs_df[["player_id", "player_name"]].dropna().drop_duplicates("player_id").copy()
    players_df["birth_date"] = pd.NaT
    players_df["position"] = pd.NA
    players_df["height"] = pd.NA
    players_df["weight"] = pd.NA
    return players_df[["player_id", "player_name", "birth_date", "position", "height", "weight"]]

def position_from_bio_flags(row: pd.Series) -> str | None:
    """Input: one eoin Players.csv row. Output: compact position string derived from guard/forward/center flags."""
    positions = []
    if pd.to_numeric(row.get("guard"), errors="coerce") == 1:
        positions.append("G")
    if pd.to_numeric(row.get("forward"), errors="coerce") == 1:
        positions.append("F")
    if pd.to_numeric(row.get("center"), errors="coerce") == 1:
        positions.append("C")
    return "/".join(positions) if positions else None

def normalize_eoin_players(path: Path) -> pd.DataFrame:
    """Input: eoin Players.csv path. Output: canonical players table with birth date, position, height, and weight."""
    raw = pd.read_csv(path, low_memory=False)
    players_df = pd.DataFrame({
        "player_id": pd.to_numeric(raw["personId"], errors="coerce").astype("Int64"),
        "player_name": (
            raw["firstName"].fillna("").astype(str).str.strip()
            + " "
            + raw["lastName"].fillna("").astype(str).str.strip()
        ).str.strip(),
        "birth_date": pd.to_datetime(raw["birthDate"], errors="coerce"),
        "position": raw.apply(position_from_bio_flags, axis=1),
        "height": pd.to_numeric(raw["heightInches"], errors="coerce"),
        "weight": pd.to_numeric(raw["bodyWeightLbs"], errors="coerce"),
    })
    return players_df.dropna(subset=["player_id"]).drop_duplicates("player_id").reset_index(drop=True)

def needs_player_bio_refresh(path: Path) -> bool:
    """Input: canonical players path. Output: True when bio source should refresh missing profile columns."""
    if should_fetch(path):
        return True
    current = read_table(path)
    if current.empty:
        return True
    required_profile_cols = ["birth_date", "position", "height", "weight"]
    missing_cols = [column for column in required_profile_cols if column not in current.columns]
    if missing_cols:
        return True
    return current[required_profile_cols].isna().mean().mean() > 0.25

def build_season_stats_from_game_logs(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical game logs. Output: season-level baseline stats for role features."""
    if game_logs_df.empty:
        return pd.DataFrame()
    df = game_logs_df.copy()
    grouped = df.groupby(["player_id", "player_name", "season", "team_id"], dropna=False)
    agg_spec = {
        "games": ("game_id", "nunique"),
        "minutes": ("minutes", "sum"),
        "points": ("points", "sum"),
        "assists": ("assists", "sum"),
        "rebounds": ("rebounds", "sum"),
    }
    for optional_column in [
        "turnovers", "fga", "fg3a", "fta", "steals", "blocks", "defensive_rebounds", "personal_fouls",
    ]:
        if optional_column in df.columns:
            agg_spec[optional_column] = (optional_column, "sum")
    season = grouped.agg(**agg_spec).reset_index()
    for optional_column in [
        "turnovers", "fga", "fg3a", "fta", "steals", "blocks", "defensive_rebounds", "personal_fouls",
    ]:
        if optional_column not in season.columns:
            season[optional_column] = np.nan
    per_100_factor = 100 / season["minutes"].replace(0, np.nan)
    season["age"] = pd.NA
    season["usage_pct"] = pd.NA
    season["points_per_100"] = season["points"] * per_100_factor
    season["assists_per_100"] = season["assists"] * per_100_factor
    season["rebounds_per_100"] = season["rebounds"] * per_100_factor
    denom = 2 * (season["fga"] + 0.44 * season["fta"]).replace(0, np.nan)
    season["true_shooting_pct"] = season["points"] / denom
    season["three_point_attempt_rate"] = season["fg3a"] / season["fga"].replace(0, np.nan)
    season["free_throw_rate"] = season["fta"] / season["fga"].replace(0, np.nan)
    season["turnover_rate"] = season["turnovers"] / (season["fga"] + 0.44 * season["fta"] + season["turnovers"]).replace(0, np.nan)
    per_36_factor = 36 / season["minutes"].replace(0, np.nan)
    season["steal_rate"] = season["steals"] * per_36_factor
    season["block_rate"] = season["blocks"] * per_36_factor
    season["defensive_rebound_rate"] = season["defensive_rebounds"] * per_36_factor
    season["foul_rate"] = season["personal_fouls"] * per_36_factor
    season["pace"] = pd.NA
    season["possessions"] = pd.NA
    season["offensive_rating"] = pd.NA
    season["defensive_rating"] = pd.NA
    return season[["player_id", "player_name", "season", "team_id", "age", "minutes", "usage_pct", "points_per_100", "assists_per_100", "rebounds_per_100", "true_shooting_pct", "three_point_attempt_rate", "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate", "foul_rate", "pace", "possessions", "offensive_rating", "defensive_rating"]]

def normalize_thomas_player_season_stats(
    advanced_path: Path,
    usage_path: Path,
    defense_path: Path,
) -> pd.DataFrame:
    """Input: Thomas advanced/usage/defense CSV paths. Output: canonical player-season advanced feature table."""
    if not advanced_path.exists():
        return pd.DataFrame()

    advanced = pd.read_csv(advanced_path, low_memory=False)
    merge_keys = ["PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "AGE", "MIN", "SEASON"]
    source = advanced.copy()

    if usage_path.exists():
        usage = pd.read_csv(usage_path, low_memory=False)
        usage_keep = [column for column in [*merge_keys, "USG_PCT", "PCT_STL", "PCT_BLK"] if column in usage.columns]
        source = source.merge(usage[usage_keep], on=merge_keys, how="left", suffixes=("", "_USAGE"))

    if defense_path.exists():
        defense = pd.read_csv(defense_path, low_memory=False)
        defense_keep = [column for column in [*merge_keys, "DEF_RATING", "DREB_PCT", "STL", "PCT_STL", "BLK", "PCT_BLK"] if column in defense.columns]
        source = source.merge(defense[defense_keep], on=merge_keys, how="left", suffixes=("", "_DEFENSE"))

    def merged_column(name: str, fallback: object = np.nan) -> pd.Series:
        """Input: canonical source column name. Output: first non-null matching Series across merged suffixes."""
        candidates = [name, f"{name}_USAGE", f"{name}_DEFENSE"]
        existing = [column for column in candidates if column in source.columns]
        if not existing:
            return pd.Series(fallback, index=source.index)
        result = source[existing[0]]
        for column in existing[1:]:
            result = result.fillna(source[column])
        return result

    normalized = pd.DataFrame({
        "player_id": pd.to_numeric(source["PLAYER_ID"], errors="coerce").astype("Int64"),
        "player_name": source["PLAYER_NAME"],
        "season": source["SEASON"],
        "team_id": source["TEAM_ABBREVIATION"],
        "team_nba_id": source["TEAM_ID"],
        "team_abbreviation": source["TEAM_ABBREVIATION"],
        "age": pd.to_numeric(source["AGE"], errors="coerce"),
        "minutes": pd.to_numeric(source["MIN"], errors="coerce"),
        "usage_pct": pd.to_numeric(merged_column("USG_PCT"), errors="coerce"),
        "true_shooting_pct": pd.to_numeric(source.get("TS_PCT"), errors="coerce"),
        "turnover_rate": pd.to_numeric(source.get("TM_TOV_PCT"), errors="coerce"),
        "steal_rate": pd.to_numeric(merged_column("PCT_STL"), errors="coerce"),
        "block_rate": pd.to_numeric(merged_column("PCT_BLK"), errors="coerce"),
        "defensive_rebound_rate": pd.to_numeric(merged_column("DREB_PCT"), errors="coerce"),
        "pace": pd.to_numeric(source.get("PACE"), errors="coerce"),
        "possessions": pd.to_numeric(source.get("POSS"), errors="coerce"),
        "offensive_rating": pd.to_numeric(source.get("OFF_RATING"), errors="coerce"),
        "defensive_rating": pd.to_numeric(merged_column("DEF_RATING"), errors="coerce"),
    })
    return normalized.dropna(subset=["player_id"]).drop_duplicates(["player_id", "season", "team_id"]).reset_index(drop=True)

def player_id_lookup_from_players(players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical players table. Output: player name key to NBA player_id lookup."""
    if players_df.empty or "player_name" not in players_df.columns:
        return pd.DataFrame(columns=["player_name_key", "player_id"])
    lookup = players_df[["player_id", "player_name"]].dropna().drop_duplicates("player_id").copy()
    lookup["player_name_key"] = lookup["player_name"].map(normalize_name_key)
    return lookup[["player_name_key", "player_id"]].dropna().drop_duplicates("player_name_key")

def normalize_rodney_advanced_patch(path: Path, players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: Rodney Basketball-Reference advanced CSV and players. Output: 2023-24 advanced patch table."""
    if not path.exists():
        return pd.DataFrame()
    raw = pd.read_csv(path, low_memory=False)
    raw = raw[(raw["lg"].eq("NBA")) & (raw["season"].eq(2024)) & (~raw["tm"].eq("TOT"))].copy()
    raw["player_name_key"] = raw["player"].map(normalize_name_key)
    raw = raw.merge(player_id_lookup_from_players(players_df), on="player_name_key", how="left")
    normalized = pd.DataFrame({
        "player_id": raw["player_id_y"],
        "player_name": raw["player"],
        "season": raw["season"].map(season_label_from_bref_end_year),
        "team_id": raw["tm"].map(normalize_team_abbreviation),
        "age": pd.to_numeric(raw["age"], errors="coerce"),
        "minutes": pd.to_numeric(raw["mp"], errors="coerce"),
        "usage_pct": pd.to_numeric(raw["usg_percent"], errors="coerce"),
        "true_shooting_pct": pd.to_numeric(raw["ts_percent"], errors="coerce"),
        "three_point_attempt_rate": pd.to_numeric(raw["x3p_ar"], errors="coerce"),
        "free_throw_rate": pd.to_numeric(raw["f_tr"], errors="coerce"),
        "turnover_rate": pd.to_numeric(raw["tov_percent"], errors="coerce"),
        "steal_rate": pd.to_numeric(raw["stl_percent"], errors="coerce"),
        "block_rate": pd.to_numeric(raw["blk_percent"], errors="coerce"),
        "defensive_rebound_rate": pd.to_numeric(raw["drb_percent"], errors="coerce"),
        "offensive_rating": pd.NA,
        "defensive_rating": pd.NA,
    })
    return normalized.dropna(subset=["player_id"]).drop_duplicates(["player_id", "season", "team_id"]).reset_index(drop=True)

def normalize_ratin_2024_25_advanced_patch(
    advanced_path: Path,
    totals_path: Path,
    players_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: Ratin21 2024-25 advanced/totals CSVs and players. Output: 2024-25 advanced patch table."""
    if not advanced_path.exists():
        return pd.DataFrame()
    advanced = pd.read_csv(advanced_path, low_memory=False)
    if totals_path.exists():
        totals = pd.read_csv(totals_path, low_memory=False)
        raw = advanced.merge(totals[["Player", "Age", "Team", "Pos", "MP"]], on="Player", how="left")
    else:
        raw = advanced.copy()
        raw["Age"] = pd.NA
        raw["Team"] = pd.NA
        raw["MP"] = pd.NA

    raw["player_name_key"] = raw["Player"].map(normalize_name_key)
    raw = raw.merge(player_id_lookup_from_players(players_df), on="player_name_key", how="left")
    normalized = pd.DataFrame({
        "player_id": raw["player_id"],
        "player_name": raw["Player"],
        "season": "2024-25",
        "team_id": raw["Team"].map(normalize_team_abbreviation),
        "age": pd.to_numeric(raw["Age"], errors="coerce"),
        "minutes": pd.to_numeric(raw["MP"], errors="coerce"),
        "usage_pct": pd.to_numeric(raw["USG%"], errors="coerce"),
        "true_shooting_pct": pd.to_numeric(raw["TS%"], errors="coerce"),
        "offensive_rating": pd.NA,
        "defensive_rating": pd.NA,
    })
    return normalized.dropna(subset=["player_id"]).drop_duplicates(["player_id", "season", "team_id"]).reset_index(drop=True)

def build_advanced_patch_stats(players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: players table. Output: combined 2023-24 and 2024-25 advanced patch stats."""
    frames = [
        normalize_rodney_advanced_patch(RODNEY_ADVANCED_PATH, players_df),
        normalize_ratin_2024_25_advanced_patch(RATIN_ADVANCED_2024_25_PATH, RATIN_TOTAL_2024_25_PATH, players_df),
    ]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def enrich_season_stats_with_advanced(
    baseline_df: pd.DataFrame,
    advanced_df: pd.DataFrame,
    players_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: baseline box-score stats, advanced stats, players. Output: enriched player-season stats table."""
    if baseline_df.empty:
        return advanced_df
    if advanced_df.empty:
        return baseline_df

    enriched = baseline_df.merge(
        advanced_df,
        on=["player_id", "season", "team_id"],
        how="left",
        suffixes=("", "_advanced"),
    )

    fallback_cols = [
        "player_id", "season", "age", "minutes", "usage_pct", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate",
        "pace", "possessions", "offensive_rating", "defensive_rating",
    ]
    fallback_cols = [column for column in fallback_cols if column in advanced_df.columns]
    if {"player_id", "season"}.issubset(fallback_cols):
        fallback = (
            advanced_df[fallback_cols]
            .sort_values(["player_id", "season", "minutes" if "minutes" in advanced_df.columns else "player_id"], ascending=[True, True, False])
            .drop_duplicates(["player_id", "season"])
        )
        enriched = enriched.merge(fallback, on=["player_id", "season"], how="left", suffixes=("", "_fallback"))

    for column in [
        "player_name", "age", "minutes", "usage_pct", "true_shooting_pct", "three_point_attempt_rate",
        "free_throw_rate", "turnover_rate", "steal_rate", "block_rate", "defensive_rebound_rate",
        "pace", "possessions", "offensive_rating", "defensive_rating",
    ]:
        advanced_column = f"{column}_advanced"
        fallback_column = f"{column}_fallback"
        if advanced_column in enriched.columns:
            enriched[column] = enriched[column].fillna(enriched[advanced_column])
        if fallback_column in enriched.columns:
            enriched[column] = enriched[column].fillna(enriched[fallback_column])

    if not players_df.empty and "birth_date" in players_df.columns:
        bio = players_df[["player_id", "birth_date"]].dropna().drop_duplicates("player_id")
        enriched = enriched.merge(bio, on="player_id", how="left")
        season_start = enriched["season"].astype(str).str.slice(0, 4).astype(float)
        birth_year = pd.to_datetime(enriched["birth_date"], errors="coerce").dt.year
        enriched["age"] = enriched["age"].fillna(season_start - birth_year)
        enriched = enriched.drop(columns=["birth_date"])

    keep_cols = [
        "player_id", "player_name", "season", "team_id", "age", "minutes", "usage_pct",
        "points_per_100", "assists_per_100", "rebounds_per_100", "true_shooting_pct",
        "three_point_attempt_rate", "free_throw_rate", "turnover_rate", "steal_rate", "block_rate",
        "defensive_rebound_rate", "foul_rate", "pace", "possessions", "offensive_rating", "defensive_rating",
    ]
    keep_cols = [column for column in keep_cols if column in enriched.columns]
    return enriched[keep_cols].sort_values(["season", "player_id", "team_id"]).reset_index(drop=True)

def needs_advanced_refresh(path: Path) -> bool:
    """Input: canonical season stats path. Output: True when advanced source should refresh sparse rate columns."""
    if should_fetch(path):
        return True
    current = read_table(path)
    if current.empty:
        return True
    target_patch_seasons = ["2023-24", "2024-25"]
    if "season" in current.columns and "usage_pct" in current.columns:
        target_rows = current[current["season"].isin(target_patch_seasons)]
        if not target_rows.empty and target_rows["usage_pct"].isna().mean() > 0.25:
            print("Refreshing season stats because advanced patch seasons are still sparse.")
            return True
    key_cols = ["usage_pct", "defensive_rebound_rate", "offensive_rating", "defensive_rating"]
    existing = [column for column in key_cols if column in current.columns]
    if len(existing) < len(key_cols):
        return True
    return current[existing].isna().mean().mean() > 0.25

def season_start_year(season: str) -> int:
    """Input: season label such as 2024-25. Output: starting year as integer."""
    return int(str(season).split("-")[0])

def cache_safe_season(season: str) -> str:
    """Input: season label. Output: filesystem-safe token used in cache file names."""
    return str(season).replace("-", "_")

def nba_api_get_data_frame(endpoint_factory, label: str) -> pd.DataFrame:
    """Input: nba_api endpoint factory and label. Output: first DataFrame with timeout retries."""
    import time
    from requests.exceptions import ReadTimeout, Timeout

    last_error: Exception | None = None
    for attempt in range(1, NBA_API_MAX_RETRIES + 1):
        try:
            endpoint = endpoint_factory()
            return endpoint.get_data_frames()[0]
        except (ReadTimeout, Timeout, TimeoutError) as exc:
            last_error = exc
            wait_seconds = REQUEST_SLEEP_SECONDS * attempt
            print(f"Timeout while fetching {label}; retry {attempt}/{NBA_API_MAX_RETRIES} after {wait_seconds:.0f}s")
            time.sleep(wait_seconds)

    print(f"Failed to fetch {label}: {last_error}")
    return pd.DataFrame()

def normalize_nba_players() -> pd.DataFrame:
    """Input: none. Output: minimal players table from nba_api static player list."""
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    raw_players = pd.DataFrame(nba_static_players.get_players())
    players_df = raw_players.rename(
        columns={
            "id": "player_id",
            "full_name": "player_name",
        }
    )
    players_df["birth_date"] = pd.NaT
    players_df["position"] = pd.NA
    players_df["height"] = pd.NA
    players_df["weight"] = pd.NA
    return players_df[["player_id", "player_name", "birth_date", "position", "height", "weight"]]

def normalize_player_game_logs(logs: pd.DataFrame) -> pd.DataFrame:
    """Input: raw nba_api PlayerGameLogs DataFrame. Output: canonical player_game_logs DataFrame."""
    if logs.empty:
        return pd.DataFrame()

    normalized = pd.DataFrame({
        "player_id": logs["PLAYER_ID"],
        "player_name": logs.get("PLAYER_NAME"),
        "game_date": pd.to_datetime(logs["GAME_DATE"]),
        "game_id": logs["GAME_ID"],
        "season": logs["REQUESTED_SEASON"],
        "team_id": logs["TEAM_ID"],
        "team_abbreviation": logs.get("TEAM_ABBREVIATION"),
        "minutes": logs["MIN"],
        "points": logs["PTS"],
        "assists": logs["AST"],
        "rebounds": logs["REB"],
        "usage_pct": pd.NA,
        "true_shooting_pct": pd.NA,
        "opponent": logs["MATCHUP"].astype(str).str.extract(r"(?:vs\.|@)\s+([A-Z]{2,3})", expand=False),
        "home_away": np.where(logs["MATCHUP"].astype(str).str.contains(" @ ", regex=False), "AWAY", "HOME"),
    })
    normalized = normalized.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    normalized["rest_days"] = normalized.groupby("player_id")["game_date"].diff().dt.days
    return normalized

def fetch_league_player_game_logs(seasons: list[str]) -> pd.DataFrame:
    """Input: season labels. Output: combined nba_api game logs; fallback only, not primary source."""
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    frames = []
    for season in seasons:
        cache_path = NBA_API_CACHE_DIR / f"player_game_logs_{cache_safe_season(season)}.parquet"
        if cache_path.exists() and not OVERWRITE_EXISTING_NBA_API_RAW:
            print(f"Using cached player game logs: {season}")
            frames.append(pd.read_parquet(cache_path))
            continue

        print(f"Fetching player game logs: {season}")
        raw = nba_api_get_data_frame(
            lambda: playergamelogs.PlayerGameLogs(
                season_nullable=season,
                season_type_nullable=SEASON_TYPE,
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player game logs {season}",
        )
        if raw.empty:
            continue
        raw["REQUESTED_SEASON"] = season
        raw.to_parquet(cache_path, index=False)
        frames.append(raw)

    if not frames:
        return pd.DataFrame()

    return normalize_player_game_logs(pd.concat(frames, ignore_index=True))

def normalize_player_season_stats(stats: pd.DataFrame) -> pd.DataFrame:
    """Input: raw nba_api season stats. Output: canonical season_stats DataFrame."""
    if stats.empty:
        return pd.DataFrame()

    def col(name: str, default: float | None = np.nan) -> pd.Series:
        """Input: column name/default. Output: existing column or default-valued Series."""
        if name in stats.columns:
            return stats[name]
        return pd.Series(default, index=stats.index)

    fga = col("FGA").replace(0, np.nan)
    normalized = pd.DataFrame({
        "player_id": stats["PLAYER_ID"],
        "player_name": col("PLAYER_NAME", pd.NA),
        "season": stats["REQUESTED_SEASON"],
        "team_id": stats["TEAM_ID"],
        "age": col("AGE"),
        "minutes": col("MIN"),
        "usage_pct": col("USG_PCT"),
        "points_per_100": col("PTS"),
        "assists_per_100": col("AST"),
        "rebounds_per_100": col("REB"),
        "true_shooting_pct": col("TS_PCT"),
        "three_point_attempt_rate": col("FG3A") / fga,
        "free_throw_rate": col("FTA") / fga,
        "turnover_rate": col("TM_TOV_PCT").fillna(col("TOV_PCT")),
        "steal_rate": col("STL_PCT"),
        "block_rate": col("BLK_PCT"),
        "offensive_rating": col("OFF_RATING"),
        "defensive_rating": col("DEF_RATING"),
    })
    return normalized.sort_values(["season", "player_id"]).reset_index(drop=True)

def fetch_league_player_season_stats(seasons: list[str]) -> pd.DataFrame:
    """Input: season labels. Output: combined nba_api season stats; fallback only."""
    if not NBA_API_AVAILABLE:
        return pd.DataFrame()

    frames = []
    for season in seasons:
        cache_path = NBA_API_CACHE_DIR / f"player_season_stats_{cache_safe_season(season)}.parquet"
        if cache_path.exists() and not OVERWRITE_EXISTING_NBA_API_RAW:
            print(f"Using cached player season stats: {season}")
            frames.append(pd.read_parquet(cache_path))
            continue

        print(f"Fetching player season stats: {season}")
        base = nba_api_get_data_frame(
            lambda: leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star=SEASON_TYPE,
                per_mode_detailed="Per100Possessions",
                measure_type_detailed_defense="Base",
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player season base stats {season}",
        )
        advanced = nba_api_get_data_frame(
            lambda: leaguedashplayerstats.LeagueDashPlayerStats(
                season=season,
                season_type_all_star=SEASON_TYPE,
                per_mode_detailed="Per100Possessions",
                measure_type_detailed_defense="Advanced",
                timeout=NBA_API_TIMEOUT_SECONDS,
            ),
            label=f"player season advanced stats {season}",
        )
        if base.empty or advanced.empty:
            continue
        merge_keys = [key for key in ["PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "AGE"] if key in base.columns and key in advanced.columns]
        raw = base.merge(advanced, on=merge_keys, how="left", suffixes=("", "_ADV"))
        raw["REQUESTED_SEASON"] = season
        raw.to_parquet(cache_path, index=False)
        frames.append(raw)

    if not frames:
        return pd.DataFrame()

    return normalize_player_season_stats(pd.concat(frames, ignore_index=True))


In [ ]:
def should_fetch(path: Path) -> bool:
    """Input: target path. Output: True when file is missing or overwrite flag is enabled."""
    return OVERWRITE_EXISTING_NBA_API_RAW or not path.exists()

def needs_game_logs_refresh(path: Path) -> bool:
    """Input: canonical game logs path. Output: True when cached logs miss required context/box-score columns."""
    if should_fetch(path):
        return True
    current_columns = set(pd.read_parquet(path, columns=None).columns)
    required_columns = {
        "opponent", "home_away", "steals", "blocks", "defensive_rebounds", "true_shooting_pct",
    }
    missing_columns = sorted(required_columns - current_columns)
    if missing_columns:
        print("Refreshing game logs because cached file is missing:", missing_columns)
        return True
    return False

if AUTO_FETCH_KAGGLE_TRADITIONAL_IF_MISSING and needs_game_logs_refresh(GAME_LOGS_PATH):
    data_path = NBA_TRADITIONAL_DIR
    if not any(data_path.rglob("*")):
        data_path = download_nba_traditional_dataset()

    player_boxscore_path = find_player_boxscore_file(data_path)
    print("Selected player box-score file:", player_boxscore_path)
    player_game_logs_raw = read_tabular_file(player_boxscore_path)
    fetched_game_logs = canonicalize_kaggle_player_game_logs(player_game_logs_raw)
    fetched_game_logs.to_parquet(GAME_LOGS_PATH, index=False)
    print(f"Saved {GAME_LOGS_PATH}: {fetched_game_logs.shape}")
elif GAME_LOGS_PATH.exists():
    print(f"Using existing {GAME_LOGS_PATH}")

if PLAYER_BIO_PATH.exists() and needs_player_bio_refresh(PLAYERS_PATH):
    players_from_bio = normalize_eoin_players(PLAYER_BIO_PATH)
    players_from_bio.to_parquet(PLAYERS_PATH, index=False)
    print(f"Saved {PLAYERS_PATH} from bio source: {players_from_bio.shape}")
elif should_fetch(PLAYERS_PATH) and GAME_LOGS_PATH.exists():
    players_from_logs = build_players_from_game_logs(pd.read_parquet(GAME_LOGS_PATH))
    if not players_from_logs.empty:
        players_from_logs.to_parquet(PLAYERS_PATH, index=False)
        print(f"Saved {PLAYERS_PATH}: {players_from_logs.shape}")

if AUTO_FETCH_ADVANCED_PATCH_IF_MISSING and not RODNEY_ADVANCED_PATH.exists():
    download_kaggle_dataset(KAGGLE_ADVANCED_2023_DATASET, ADVANCED_PATCH_2023_DIR)
if AUTO_FETCH_ADVANCED_PATCH_IF_MISSING and not RATIN_ADVANCED_2024_25_PATH.exists():
    download_kaggle_dataset(KAGGLE_ADVANCED_2024_DATASET, ADVANCED_PATCH_2024_DIR)

if needs_advanced_refresh(SEASON_STATS_PATH) and GAME_LOGS_PATH.exists():
    players_for_age = read_table(PLAYERS_PATH)
    season_stats_from_logs = build_season_stats_from_game_logs(pd.read_parquet(GAME_LOGS_PATH))
    advanced_frames = [
        normalize_thomas_player_season_stats(ADVANCED_STATS_PATH, USAGE_STATS_PATH, DEFENSE_STATS_PATH),
        build_advanced_patch_stats(players_for_age),
    ]
    advanced_frames = [frame for frame in advanced_frames if not frame.empty]
    advanced_season_stats = pd.concat(advanced_frames, ignore_index=True) if advanced_frames else pd.DataFrame()
    season_stats_enriched = enrich_season_stats_with_advanced(season_stats_from_logs, advanced_season_stats, players_for_age)
    if not season_stats_enriched.empty:
        season_stats_enriched.to_parquet(SEASON_STATS_PATH, index=False)
        print(f"Saved {SEASON_STATS_PATH}: {season_stats_enriched.shape}")

# Fallback only. Historical game logs should prefer KaggleHub snapshots over live stats.nba.com requests.
if AUTO_FETCH_NBA_API_IF_MISSING:
    if not NBA_API_AVAILABLE:
        raise ImportError("Install nba_api first by running `%pip install nba-api`, then restart/re-run the notebook.")

    if should_fetch(PLAYERS_PATH):
        fetched_players = normalize_nba_players()
        fetched_players.to_parquet(PLAYERS_PATH, index=False)
        print(f"Saved {PLAYERS_PATH}: {fetched_players.shape}")
    else:
        print(f"Using existing {PLAYERS_PATH}")

    if should_fetch(NBA_API_GAME_LOGS_PATH):
        fetched_game_logs = fetch_league_player_game_logs(FETCH_SEASONS)
        fetched_game_logs.to_parquet(NBA_API_GAME_LOGS_PATH, index=False)
        print(f"Saved {NBA_API_GAME_LOGS_PATH}: {fetched_game_logs.shape}")
    else:
        print(f"Using existing {NBA_API_GAME_LOGS_PATH}")

    if should_fetch(SEASON_STATS_PATH):
        fetched_season_stats = fetch_league_player_season_stats(FETCH_SEASONS)
        fetched_season_stats.to_parquet(SEASON_STATS_PATH, index=False)
        print(f"Saved {SEASON_STATS_PATH}: {fetched_season_stats.shape}")
    else:
        print(f"Using existing {SEASON_STATS_PATH}")

missing_salary_sources = [path for path in [PLAYER_SEASON_SALARIES_PATH, SALARY_CAP_PATH] if not path.exists()]
if missing_salary_sources:
    print("Additional non-nba_api data status for salary analysis:")
    for path in missing_salary_sources:
        print(f"  {path}")


In [ ]:
players = read_table(PLAYERS_PATH)
game_logs = read_table(GAME_LOGS_PATH)
season_stats = read_table(SEASON_STATS_PATH)
salary_cap = read_table(SALARY_CAP_PATH)
player_season_salaries = read_table(PLAYER_SEASON_SALARIES_PATH)

def build_default_salary_cap_by_season() -> pd.DataFrame:
    """Input: none. Output: curated NBA salary-cap table used when Drive raw cap data is incomplete."""
    rows = [
        ("1999-00", 34000000, None),
        ("2000-01", 35500000, None),
        ("2001-02", 42500000, None),
        ("2002-03", 40271000, 52880000),
        ("2003-04", 43840000, 54556722),
        ("2004-05", 43870000, None),
        ("2005-06", 49500000, 61700000),
        ("2006-07", 53135000, 65420000),
        ("2007-08", 55630000, 67865000),
        ("2008-09", 58680000, 71150000),
        ("2009-10", 57700000, 69920000),
        ("2010-11", 58044000, 70307000),
        ("2011-12", 58044000, 70307000),
        ("2012-13", 58044000, 70307000),
        ("2013-14", 58679000, 71748000),
        ("2014-15", 63065000, 76829000),
        ("2015-16", 70000000, 84740000),
        ("2016-17", 94143000, 113287000),
        ("2017-18", 99093000, 119266000),
        ("2018-19", 101869000, 123733000),
        ("2019-20", 109140000, 132627000),
        ("2020-21", 109140000, 132627000),
        ("2021-22", 112414000, 136606000),
        ("2022-23", 123655000, 150267000),
        ("2023-24", 136021000, 165294000),
        ("2024-25", 140588000, 170814000),
        ("2025-26", 154647000, 187895000),
    ]
    return pd.DataFrame(rows, columns=["season", "salary_cap_usd", "tax_level_usd"]).assign(source="curated_salary_cap_history")

def complete_salary_cap_table(salary_cap_df: pd.DataFrame) -> pd.DataFrame:
    """Input: raw salary-cap table. Output: complete canonical salary-cap table for all salary seasons."""
    default_cap = build_default_salary_cap_by_season()
    if salary_cap_df.empty:
        cap = default_cap
    else:
        cap = salary_cap_df.copy()
        if "salary_cap" in cap.columns and "salary_cap_usd" not in cap.columns:
            cap = cap.rename(columns={"salary_cap": "salary_cap_usd"})
        for column in ["tax_level_usd", "source"]:
            if column not in cap.columns:
                cap[column] = pd.NA
        cap = pd.concat([default_cap, cap[["season", "salary_cap_usd", "tax_level_usd", "source"]]], ignore_index=True)
        cap = cap.drop_duplicates("season", keep="last")
    cap["season_start_year"] = cap["season"].astype(str).str.slice(0, 4).astype(int)
    return cap.sort_values("season_start_year").drop(columns=["season_start_year"]).reset_index(drop=True)

salary_cap = complete_salary_cap_table(salary_cap)

tables = {
    "players": players,
    "player_game_logs": game_logs,
    "player_season_stats": season_stats,
    "salary_cap_by_season": salary_cap,
    "player_season_salaries": player_season_salaries,
}

for table_name, table_df in tables.items():
    report_schema(table_name, table_df, RAW_SCHEMAS[table_name])


## 3. Gold Dataset 1: Player Role Features

Một dòng = một cầu thủ trong một mùa. Dataset này phục vụ player similarity, candidate retrieval, role explanation, và có thể tái sử dụng cho salary model.

In [ ]:
ROLE_BASE_FEATURES = [
    "minutes",
    "usage_pct",
    "points_per_100",
    "assists_per_100",
    "rebounds_per_100",
    "true_shooting_pct",
    "three_point_attempt_rate",
    "free_throw_rate",
    "turnover_rate",
    "steal_rate",
    "block_rate",
    "defensive_rebound_rate",
    "foul_rate",
    "pace",
    "possessions",
    "offensive_rating",
    "defensive_rating",
]

def add_role_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    """Input: season stats with base role features. Output: DataFrame with derived role dimensions."""
    role = df.copy()
    role["scoring_creation"] = role[["points_per_100", "usage_pct", "free_throw_rate"]].mean(axis=1)
    role["playmaking"] = role[["assists_per_100", "usage_pct"]].mean(axis=1) - role["turnover_rate"].fillna(0)
    role["shooting"] = role[["true_shooting_pct", "three_point_attempt_rate"]].mean(axis=1)
    role["rim_pressure"] = role[["free_throw_rate", "points_per_100"]].mean(axis=1)
    role["rebounding"] = role[["rebounds_per_100", "defensive_rebound_rate"]].mean(axis=1)
    role["perimeter_defense"] = role["steal_rate"]
    role["interior_defense"] = role[["block_rate", "defensive_rebound_rate"]].mean(axis=1)
    role["two_way_impact"] = role["offensive_rating"] - role["defensive_rating"]
    return role

ROLE_DIMENSIONS = [
    "scoring_creation", "playmaking", "shooting", "rim_pressure", "rebounding",
    "perimeter_defense", "interior_defense", "two_way_impact",
]

def build_player_role_features(season_stats_df: pd.DataFrame, players_df: pd.DataFrame) -> pd.DataFrame:
    """Input: season_stats and players. Output: player-season role feature table for similarity."""
    if season_stats_df.empty:
        return pd.DataFrame()

    required = ["player_id", "season", "team_id", "age"] + ROLE_BASE_FEATURES
    role = season_stats_df[required].copy()
    role = add_role_dimensions(role)

    identity_cols = [col for col in ["player_id", "player_name", "position"] if col in players_df.columns]
    if identity_cols:
        role = role.merge(players_df[identity_cols].drop_duplicates("player_id"), on="player_id", how="left")

    output_cols = [
        "player_id", "player_name", "season", "team_id", "age", "position",
        *ROLE_BASE_FEATURES,
        *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in role.columns]
    return role[output_cols].sort_values(["season", "player_id"]).reset_index(drop=True)

player_role_features = build_player_role_features(season_stats, players)
player_role_features.head()


In [ ]:
if not player_role_features.empty:
    player_role_features.to_parquet(OUTPUT_ROLE_FEATURES, index=False)
    print(f"Saved {OUTPUT_ROLE_FEATURES} with shape {player_role_features.shape}")

### Quick Similarity Prototype

Cell này giúp kiểm tra nhanh câu hỏi: cầu thủ nào có style/role gần một cầu thủ target. Có thể đổi `TARGET_PLAYER_NAME`, `TARGET_SEASON`, và `SIMILARITY_WEIGHTS`.

In [ ]:
TARGET_PLAYER_NAME = "LeBron James"
TARGET_SEASON = None

SIMILARITY_WEIGHTS = {
    "scoring_creation": 1.0,
    "playmaking": 1.2,
    "shooting": 0.8,
    "rim_pressure": 1.0,
    "rebounding": 0.7,
    "perimeter_defense": 0.8,
    "interior_defense": 0.4,
    "two_way_impact": 0.8,
}

def find_similar_players(
    role_df: pd.DataFrame,
    target_player_name: str,
    target_season: str | int | None = None,
    top_k: int = 10,
    weights: dict[str, float] | None = None,
) -> pd.DataFrame:
    """Input: role table, target player/season, top_k, weights. Output: top similar player-seasons."""
    if role_df.empty:
        return pd.DataFrame()

    features = [feature for feature in ROLE_DIMENSIONS if feature in role_df.columns]
    matrix = role_df[features].copy()
    if weights:
        for feature, weight in weights.items():
            if feature in matrix.columns:
                matrix[feature] = matrix[feature] * weight

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X = preprocessor.fit_transform(matrix)

    target_mask = role_df["player_name"].str.lower().eq(target_player_name.lower())
    if target_season is not None:
        target_mask &= role_df["season"].eq(target_season)
    if not target_mask.any():
        raise ValueError(f"Target player not found: {target_player_name}, season={target_season}")

    target_idx = role_df[target_mask].index[-1]
    sims = cosine_similarity(X[target_idx : target_idx + 1], X).ravel()

    results = role_df.copy()
    results["similarity_score"] = sims
    results = results.loc[results.index != target_idx]
    return results.sort_values("similarity_score", ascending=False).head(top_k)

if not player_role_features.empty and "player_name" in player_role_features.columns:
    similar_players = find_similar_players(
        player_role_features,
        TARGET_PLAYER_NAME,
        target_season=TARGET_SEASON,
        top_k=10,
        weights=SIMILARITY_WEIGHTS,
    )
    display(similar_players[["player_name", "season", "team_id", "position", "similarity_score", *ROLE_DIMENSIONS]])

## 4. Gold Dataset 2: Performance Training

Một dòng = một cầu thủ tại một `as_of_date`. Features chỉ dùng dữ liệu trước hoặc tại `as_of_date`; targets dùng trung bình 5 trận tiếp theo.

In [ ]:
def add_season_context_to_game_logs(
    game_logs_df: pd.DataFrame,
    season_stats_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: game logs and season stats. Output: game logs enriched with season-level usage/rate context when missing."""
    if game_logs_df.empty or season_stats_df.empty:
        return game_logs_df

    context_cols = [
        "player_id", "season", "team_id", "usage_pct", "true_shooting_pct", "pace", "possessions",
        "offensive_rating", "defensive_rating",
    ]
    context_cols = [column for column in context_cols if column in season_stats_df.columns]
    context = season_stats_df[context_cols].drop_duplicates(["player_id", "season", "team_id"])
    enriched = game_logs_df.merge(context, on=["player_id", "season", "team_id"], how="left", suffixes=("", "_season"))

    for column in ["usage_pct", "true_shooting_pct", "pace", "possessions", "offensive_rating", "defensive_rating"]:
        season_column = f"{column}_season"
        if season_column in enriched.columns:
            if column in enriched.columns:
                enriched[column] = enriched[column].fillna(enriched[season_column])
            else:
                enriched[column] = enriched[season_column]
            enriched = enriched.drop(columns=[season_column])

    return enriched

def add_rolling_player_features(game_logs_df: pd.DataFrame) -> pd.DataFrame:
    """Input: canonical game logs. Output: per-game rolling form features and next-5-game targets."""
    if game_logs_df.empty:
        return pd.DataFrame()

    df = game_logs_df.copy()
    df["game_date"] = pd.to_datetime(df["game_date"])
    df = df.sort_values(["player_id", "game_date", "game_id"]).reset_index(drop=True)
    grouped = df.groupby("player_id", group_keys=False)

    stat_prefixes = {"points": "pts", "assists": "ast", "rebounds": "reb"}

    for stat, prefix in stat_prefixes.items():
        for window in [5, 10, 20]:
            df[f"{prefix}_last_{window}"] = grouped[stat].transform(
                lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
            )
        df[f"{prefix}_season_to_date"] = grouped[stat].transform(
            lambda s: s.shift(1).expanding(min_periods=1).mean()
        )

    for stat in ["minutes", "usage_pct", "true_shooting_pct"]:
        if stat not in df.columns:
            print(f"Skipping rolling feature for missing column: {stat}")
            continue
        df[f"{stat}_last_5"] = grouped[stat].transform(
            lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()
        )
        df[f"{stat}_last_10"] = grouped[stat].transform(
            lambda s: s.shift(1).rolling(window=10, min_periods=1).mean()
        )
        df[f"{stat}_trend"] = df[f"{stat}_last_5"] - df[f"{stat}_last_10"]

    for stat in ["points", "assists", "rebounds"]:
        df[f"target_next_5_games_{stat}"] = grouped[stat].transform(
            lambda s: s.shift(-1).rolling(window=5, min_periods=1).mean().shift(-4)
        )

    df = df.rename(columns={"game_date": "as_of_date"})
    return df

def build_performance_training(
    game_logs_df: pd.DataFrame,
    season_stats_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Input: canonical game logs plus optional season stats. Output: performance_training table for time-series forecasting."""
    if season_stats_df is not None:
        game_logs_df = add_season_context_to_game_logs(game_logs_df, season_stats_df)

    features = add_rolling_player_features(game_logs_df)
    if features.empty:
        return pd.DataFrame()

    id_cols = [
        "player_id", "as_of_date", "game_id", "season", "team_id", "opponent", "home_away", "rest_days",
    ]
    feature_cols = [
        "pts_last_5", "pts_last_10", "pts_last_20", "pts_season_to_date",
        "ast_last_5", "ast_last_10", "ast_last_20", "ast_season_to_date",
        "reb_last_5", "reb_last_10", "reb_last_20", "reb_season_to_date",
        "minutes_last_5", "minutes_last_10", "minutes_trend",
        "usage_pct_last_5", "usage_pct_last_10", "usage_pct_trend",
        "true_shooting_pct_last_5", "true_shooting_pct_last_10", "true_shooting_pct_trend",
    ]
    target_cols = [
        "target_next_5_games_points", "target_next_5_games_assists", "target_next_5_games_rebounds",
    ]
    output_cols = [col for col in [*id_cols, *feature_cols, *target_cols] if col in features.columns]
    return features[output_cols].dropna(subset=target_cols, how="all").reset_index(drop=True)

performance_training = build_performance_training(game_logs, season_stats)
performance_training.head()


In [ ]:
if not performance_training.empty:
    performance_training.to_parquet(OUTPUT_PERFORMANCE_TRAINING, index=False)
    print(f"Saved {OUTPUT_PERFORMANCE_TRAINING} with shape {performance_training.shape}")

## 5. Gold Dataset 3: Salary Analysis / Training

Một dòng = một cầu thủ trong một mùa lương. Dataset này bỏ qua contract signing/term và tập trung vào phân tích `salary_usd` theo mùa, có thể join với role features để học quan hệ giữa production/role và salary.

Nếu có `salary_cap_by_season.csv`, notebook sẽ thêm `salary_cap_share = salary_usd / salary_cap_usd`. Salary model nên ưu tiên dự đoán `salary_cap_share`, rồi đổi ngược về USD theo cap của mùa cần phân tích.


In [ ]:
def normalize_name_for_join(value: object) -> str | None:
    """Input: player name. Output: normalized name key for fuzzy-light salary/role joins."""
    if pd.isna(value):
        return None
    return str(value).strip().lower().replace(".", "").replace("'", "")

def build_salary_training(
    player_salaries_df: pd.DataFrame,
    salary_cap_df: pd.DataFrame,
    role_features_df: pd.DataFrame,
    players_df: pd.DataFrame,
) -> pd.DataFrame:
    """Input: salaries, salary cap table, role features, players. Output: salary_training table."""
    if player_salaries_df.empty:
        return pd.DataFrame()

    salary = player_salaries_df.copy()
    salary["player_name_join"] = salary["player_name"].map(normalize_name_for_join)
    salary["target_salary_usd"] = salary["salary_usd"]

    if not salary_cap_df.empty:
        cap = salary_cap_df.copy()
        if "season" in cap.columns and "season_label" not in cap.columns:
            cap = cap.rename(columns={"season": "season_label"})
        if "salary_cap" in cap.columns and "salary_cap_usd" not in cap.columns:
            cap = cap.rename(columns={"salary_cap": "salary_cap_usd"})
        salary = salary.merge(cap[["season_label", "salary_cap_usd"]], on="season_label", how="left")
        salary["salary_cap_share"] = salary["salary_usd"] / salary["salary_cap_usd"]
    else:
        salary["salary_cap_usd"] = pd.NA
        salary["salary_cap_share"] = pd.NA

    if not role_features_df.empty and "player_name" in role_features_df.columns:
        role = role_features_df.copy()
        role["player_name_join"] = role["player_name"].map(normalize_name_for_join)
        role = role.rename(columns={"season": "season_label"})
        keep_cols = [
            "player_name_join", "season_label", "player_id", "team_id", "age", "position",
            *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS,
        ]
        keep_cols = [col for col in keep_cols if col in role.columns]
        salary = salary.merge(
            role[keep_cols].drop_duplicates(["player_name_join", "season_label"]),
            on=["player_name_join", "season_label"],
            how="left",
        )

    if not players_df.empty and "player_name" in players_df.columns:
        bio = players_df.copy()
        bio["player_name_join"] = bio["player_name"].map(normalize_name_for_join)
        bio_cols = ["player_name_join", "player_id", "birth_date", "position", "height", "weight"]
        bio_cols = [column for column in bio_cols if column in bio.columns]
        salary = salary.merge(
            bio[bio_cols].drop_duplicates("player_name_join"),
            on="player_name_join",
            how="left",
            suffixes=("", "_bio"),
        )
        if "player_id_bio" in salary.columns:
            salary["player_id"] = salary.get("player_id", pd.Series(pd.NA, index=salary.index)).fillna(salary["player_id_bio"])
        if "position_bio" in salary.columns:
            salary["position"] = salary.get("position", pd.Series(pd.NA, index=salary.index)).fillna(salary["position_bio"])
        if "age" not in salary.columns:
            salary["age"] = pd.NA
        if "birth_date" in salary.columns:
            birth_year = pd.to_datetime(salary["birth_date"], errors="coerce").dt.year
            salary["age"] = salary["age"].fillna(salary["season_start_year"] - birth_year)

    output_cols = [
        "player_id", "player_name", "team", "team_id", "season_start_year", "season_end_year",
        "season_label", "age", "position", "height", "weight", "salary_usd", "salary_cap_usd", "salary_cap_share",
        "target_salary_usd", "source", "source_file", "collected_at",
        *ROLE_BASE_FEATURES, *ROLE_DIMENSIONS,
    ]
    output_cols = [col for col in output_cols if col in salary.columns]
    return salary[output_cols].sort_values(["season_start_year", "salary_usd", "player_name"], ascending=[True, False, True]).reset_index(drop=True)

salary_training = build_salary_training(player_season_salaries, salary_cap, player_role_features, players)
salary_training.head()


In [ ]:
if not salary_training.empty:
    salary_training.to_parquet(OUTPUT_SALARY_TRAINING, index=False)
    print(f"Saved {OUTPUT_SALARY_TRAINING} with shape {salary_training.shape}")

## 6. Basic Data Quality Checks

Các check này chỉ để exploration. Khi logic ổn, có thể chuyển thành test hoặc Great Expectations suite sau.

In [ ]:
def quality_summary(name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Input: table name and DataFrame. Output: column-level dtype/missing/unique summary."""
    if df.empty:
        print(f"{name}: empty")
        return pd.DataFrame()
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_pct": df.isna().mean(),
        "n_unique": df.nunique(dropna=True),
    })
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    return summary.sort_values("missing_pct", ascending=False)

quality_summary("player_role_features", player_role_features).head(20)

In [ ]:
quality_summary("performance_training", performance_training).head(20)

In [ ]:
quality_summary("salary_training", salary_training).head(20)

## 7. Temporal Splits

Vì đây là bài toán theo thời gian, không random split. Các mùa gần nhất nên được giữ lại để validation/test/holdout, nhất là khi salary data chỉ đến `2024-25`.

In [ ]:
def assign_temporal_split(season_label: object) -> str:
    """Input: season label. Output: train/validation/test/final_holdout split label."""
    if pd.isna(season_label):
        return "unknown"
    season = str(season_label)
    if season in FINAL_HOLDOUT_SEASONS:
        return "final_holdout"
    if season in TEST_SEASONS:
        return "test"
    if season in VALIDATION_SEASONS:
        return "validation"
    if season <= TRAIN_END_SEASON:
        return "train"
    return "future_or_unassigned"

if not salary_training.empty and "season_label" in salary_training.columns:
    salary_training = salary_training.copy()
    salary_training["split"] = salary_training["season_label"].map(assign_temporal_split)
    salary_training.to_parquet(OUTPUT_SALARY_TRAINING, index=False)
    display(salary_training["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))

if not performance_training.empty and "season" in performance_training.columns:
    performance_training = performance_training.copy()
    performance_training["split"] = performance_training["season"].map(assign_temporal_split)
    performance_training.to_parquet(OUTPUT_PERFORMANCE_TRAINING, index=False)
    display(performance_training["split"].value_counts(dropna=False).rename_axis("split").reset_index(name="rows"))

## 8. Added Feature Sources

Các feature quan trọng cho replacement search và salary analysis hiện đã có đường đọc raw rõ ràng trong notebook:

- Player bio/profile: `data/raw/player_bio/eoin_players.csv` -> `players` với `birth_date`, `position`, `height`, `weight`. `age` được tính theo mùa khi build season stats.
- Advanced/rate stats: `data/raw/player_stats_advanced/player_stats_advanced_rs.csv`, `player_stats_usage_rs.csv`, `player_stats_defense_rs.csv` -> `usage_pct`, `true_shooting_pct`, `steal_rate`, `block_rate`, `defensive_rebound_rate`, `pace`, `possessions`, ratings. Nguồn chính bao phủ `1996-97` đến `2022-23`; notebook tự vá `2023-24` từ `rodneycarroll78/nba-stats-1980-2024` và `2024-25` từ `ratin21/nba-player-stats-2024-25-per-game`.
- Game context/proxy: `traditional.csv` có thể tạo `opponent`, `home_away`, `true_shooting_pct`, rolling form, và defensive per-36 proxy từ box score.
- Salary market context: `data/raw/salary_cap/salary_cap_by_season.csv` -> `salary_cap_share`.

Phần còn cần bổ sung sau: nếu cần player tracking/on-off hoặc official NBA ratings cho `2023-24` và `2024-25`; bản notebook hiện đã vá được `USG%`, `TS%`, `STL%`, `BLK%`, `DRB%` cho hai mùa này từ snapshot CSV.


## 9. Next Decisions

- Chốt có dùng thêm `damirdizdarevic` để vá `2023-24` hay tìm nguồn khác có `PLAYER_ID` sạch hơn.
- Chốt cách join salary với stats: cùng mùa để phân tích mô tả, hoặc mùa trước nếu dùng làm model dự đoán salary tương lai.
- Chốt role dimensions: công thức hiện tại chỉ là baseline heuristic để exploration, chưa phải product formula cuối.
- Chốt target horizon cho performance forecast: 5 games, 10 games, hoặc remainder-of-season.
- Giữ `2024-25` làm final holdout khi có đủ feature/salary overlap; không dùng để chọn model.
- Sau khi notebook chạy ổn, chuyển từng function thành module xử lý dữ liệu có test.
